In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from seq2seq.model.encoder import Encoder
from seq2seq.model.decoder import Decoder
from seq2seq.model.seq2seq import Seq2Seq

In [3]:
SRC_VOCAB_SIZE = 10
TGT_VOCAB_SIZE = 10

BATCH_SIZE = 4
SRC_LENGTH = 3
TGT_LENGTH  = 3

EMBEDDING_DIM = 32
HIDDEN_DIM = 64
NUM_LAYERS = 2

SOS_IDX = 1
EOS_IDX = 2

In [21]:
src = torch.randint(
    low=3,
    high=SRC_VOCAB_SIZE,
    size=(BATCH_SIZE, SRC_LENGTH)
)

tgt = src.clone().detach()

SOS_col = torch.tensor([SOS_IDX])
SOS_col = SOS_col.expand(BATCH_SIZE, -1)

EOS_col = torch.tensor([EOS_IDX])
EOS_col = EOS_col.expand(BATCH_SIZE, -1)

tgt = torch.cat((SOS_col, src, EOS_col), dim=1)

src, tgt

(tensor([[5, 8, 8],
         [6, 7, 8],
         [3, 6, 3],
         [8, 3, 3]]),
 tensor([[1, 5, 8, 8, 2],
         [1, 6, 7, 8, 2],
         [1, 3, 6, 3, 2],
         [1, 8, 3, 3, 2]]))

In [5]:
BATCH_LENGTH = src.shape[0]
SRC_LENGTH = src.shape[1]
TGT_LENGTH = tgt.shape[1]

In [6]:
encoder = Encoder(
    vocab_size=SRC_VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
)

decoder = Decoder(
    vocab_size=TGT_VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
)

model = Seq2Seq(
    encoder=encoder,
    decoder=decoder,
    target_vocab_size=TGT_VOCAB_SIZE,
)

In [ ]:
logits = model.forward(src, tgt, False)

In [9]:
targets = tgt[:, 1:]

In [10]:
loss = F.cross_entropy(
    logits.reshape(-1, TGT_VOCAB_SIZE),
    targets.reshape(-1)
)

loss.item()

2.2997453212738037

In [11]:
loss.backward()

In [12]:
assert encoder.embedding.weight.grad.abs().sum() > 0
assert decoder.output.weight.grad.abs().sum() > 0

In [13]:
print(src.shape)
print(tgt.shape)
print(targets.shape)
print(logits.shape)

torch.Size([4, 3])
torch.Size([4, 5])
torch.Size([4, 4])
torch.Size([4, 4, 10])


In [29]:
'''
Training loop:
1. Zero old gradients
2. Forward pass
3. Compute loss
4. Backward pass
5. Update weights
''';

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

def train(loops):
    for i in range(loops):
        optimizer.zero_grad()
        logits = model(src, tgt, True)
        
        loss = F.cross_entropy(
            logits.reshape(-1, TGT_VOCAB_SIZE),
            targets.reshape(-1)
        )
        
        print(loss.item())
        
        loss.backward()
        optimizer.step()

train(5)



0.5139077305793762
0.7702596783638
0.2427922487258911
0.20533061027526855
0.18823879957199097


In [32]:
model.eval()

with torch.no_grad():
    logits = model(src, tgt, False)
    predictions = logits.argmax(dim=-1)

targets = tgt[:, 1:]

print(src)
print(targets)
print(predictions)

print((predictions == targets).float().mean().item())

print((predictions == targets).all(dim=1))

tensor([[5, 8, 8],
        [6, 7, 8],
        [3, 6, 3],
        [8, 3, 3]])
tensor([[5, 8, 8, 2],
        [6, 7, 8, 2],
        [3, 6, 3, 2],
        [8, 3, 3, 2]])
tensor([[5, 8, 8, 2],
        [6, 7, 8, 2],
        [3, 6, 3, 2],
        [8, 3, 3, 2]])
1.0
tensor([True, True, True, True])
